# Task 2: Original LCS System on Raw Dataset

## Imports

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer, RobustScaler, KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from skeLCS import eLCS
import time

## Loading raw dataset

In [3]:
#loading the raw dataset for the baseline of LCS
df_raw_lcs = pd.read_csv('MattyLuriz_creditcard.csv')
df_raw_lcs.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## Minimal Processing: exact-count stratified subsample

In [4]:
#LCS compatibility requires minimal processing
# Rather than subsampling via train_test_split's stratify argument, the fraud
# rate in the raw data is used to compute exact per-class sample sizes, then
# each class is sampled independently. This keeps the subsample's fraud rate
# tied explicitly to the raw dataset's rate rather than relying on stratify
# to preserve it implicitly.
n_total = 20000
fraud_rate = df_raw_lcs['Class'].mean()
n_fraud = round(n_total * fraud_rate)
n_legit = n_total - n_fraud

fraud_sample = df_raw_lcs[df_raw_lcs['Class'] == 1].sample(n=n_fraud, random_state=42)
legit_sample = df_raw_lcs[df_raw_lcs['Class'] == 0].sample(n=n_legit, random_state=42)

df_sample = pd.concat([fraud_sample, legit_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Raw fraud rate: {fraud_rate:.4%}")
print(f"Subsample size: {df_sample.shape[0]} rows ({n_fraud} fraud, {n_legit} legitimate)")
print(f"Subsample fraud rate: {df_sample['Class'].mean():.4%}")

Raw fraud rate: 0.1727%
Subsample size: 20000 rows (35 fraud, 19965 legitimate)
Subsample fraud rate: 0.1750%


In [5]:
X = df_sample.drop(columns=['Class']).values
y = df_sample['Class'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Training set: {X_train.shape}, Fraud cases: {y_train.sum()}")
print(f"Test set: {X_test.shape}, Fraud cases: {y_test.sum()}")

Training set: (16000, 30), Fraud cases: 28
Test set: (4000, 30), Fraud cases: 7


## Run the original, unmodified eLCS baseline

In [8]:
model = eLCS(learning_iterations=10000, random_state=42)

start = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - start

preds = model.predict(X_test)

print(f"Training time: {elapsed:.2f} seconds")
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, preds):.4f}")
print(f"Precision: {precision_score(y_test, preds, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, preds, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, preds, zero_division=0):.4f}")

Training time: 31.98 seconds
Accuracy: 0.9985
Balanced Accuracy: 0.5714
Precision: 1.0000
Recall: 0.1429
F1 Score: 0.2500


### Task 2 Results: Original LCS System on Raw Dataset

An unmodified eLCS baseline was run on the unpreprocessed, uncleaned dataset. Since applying LCS rule evolution to 284,807 rows was computationally infeasible, a subsample was drawn of 20,000 rows (large enough to accommodate the original dataset's fraud rate of 0.1727% without dividing the rate at random) and split into training (16,000 rows) and test (4,000 rows) sets (with 28 and 7 frauds respectively). The subsample (including the target variable's position) was then reshuffled and all features were rearranged with the target in the first position to form an array that could be fed into the eLCS classifier without any other cleaning, scaling, or transforming (as this is to be used for the unpreprocessed baseline).

The model was ran with eLCS's unmodified, default parameters(learning_iterations = 10000, N = 1000, p_spec = 0.5, nu = 5, chi = 0.8, mu = 0.04, theta_GA = 25), training being completed at 31.98 seconds

**Baseline results**:
|  Metric  |  Value  |  
|----------|---------|  
| Accuracy | 0.9985  |  
| Balanced Accuracy | 0.5714 |  
|Precision | 1.0000  |  
| Recall   | 0.1429  |  
| F1-score | 0.2500  |  

The headline accuracy value provides no useful indication of performance here: in a problem space where less than 0.2% of the cases represent fraud, a classifier assigning "legitimate" to all cases would produce nearly the same score. The balanced accuracy score of 0.5714 gives a better, though still not exceptional reflection (just above 0.5, being the chance baseline). Most revealing are the precision/recall metrics, which tell the true story of the system: its constrained and highly selective rule population meant that whenever it predicted a transaction to be fraud, it was 100% correct in its claim (precision = 1.0). 

The price paid for this precision was its failure to generalize, capturing only 1 out of the 7 fraud cases (recall = 0.1429), leaving the remaining fraud cases in the test set undetected. 

With only 28 training fraud examples, there was not enough minority-class signal for eLCS to build a rule population generalized to cover most of the cases, even though it found one reliable signature of a fraud case. This low-recall, near-chance baseline provides a target that the pre-processing and class-imbalance-mitigation work of Tasks 3 and 4 intends to overcome.

# Task 3: Data Preprocessing and Feature Engineering

## Step 1: Missing values, duplicates, invalid values, inconsistent categories, data types

Some of these data integrity requirements were already covered in Phase 1 (Task 2 in data engineering): no missing values, all amount and time values valid, no inconsistencies with categories, because the data set is completely numerical. Also every column is assigned to the right data type. The only thing which needs to be performed now is to drop the 1081 duplicated rows detected earlier, which repeat here to ensure the notebook runs standalone starting from the raw file.

In [4]:
df_raw = pd.read_csv('MattyLuriz_creditcard.csv')
print(f"Raw shape: {df_raw.shape}")

df_clean = df_raw.drop_duplicates(keep='first').copy()
print(f"After removing duplicates: {df_clean.shape}")

Raw shape: (284807, 31)
After removing duplicates: (283726, 31)


## Step 2: Outliers

The first test was performed in Phase 1 was the one for detecting outliers based on the amount of transaction using the Z-score method, which flagged all transactions with |z| > 3 as outliers. 4,063 transactions, or 1.43% of the dataset, were identified as outliers and only 11 of those were fraud, which was similar to the overall spam rate in the data. The conclusion that can be drawn from this is that an unusually large transaction amount, by itself, is not a telling indicator of spam. The outliers were kept instead of being removed, since removing them might exclude authentic spam, and since those values are representative of real-world transactions, not bugs in the data. Phase 2 will go through the same process, without removing any transactions based on their amount.

## Step 3: Subsample, then split (leakage-safe from here on)

Running eLCS on the entire dataset – 283,726 rows – has been shown to be infeasible in Task 2, so a subsample is taken first, using the same exact-per-class-count approach to preserve the true fraud rate as in Task 2. This subsampling is an explicit row-sampling rather than a trained transform, so no leakage is possible here.

Hence, from now onwards, the separating line comes up before any process involving fitting parameters to data (like scaling, feature selection, discretization, etc.) is carried out. These processes are fitted (or trained) only on the training set and are applied to the test set using the parameters obtained from the training set.

In [5]:
n_total = 20000
fraud_rate = df_clean['Class'].mean()
n_fraud = round(n_total * fraud_rate)
n_legit = n_total - n_fraud

fraud_sample = df_clean[df_clean['Class'] == 1].sample(n=n_fraud, random_state=42)
legit_sample = df_clean[df_clean['Class'] == 0].sample(n=n_legit, random_state=42)

df_sample = pd.concat([fraud_sample, legit_sample]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Subsample size: {df_sample.shape[0]} rows ({n_fraud} fraud, {n_legit} legitimate)")

X = df_sample.drop(columns=['Class'])
y = df_sample['Class']

# Split BEFORE any fitted preprocessing
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train shape: {X_train_raw.shape}, Fraud cases: {y_train.sum()}")
print(f"Test shape: {X_test_raw.shape}, Fraud cases: {y_test.sum()}")

Subsample size: 20000 rows (33 fraud, 19967 legitimate)
Train shape: (16000, 30), Fraud cases: 26
Test shape: (4000, 30), Fraud cases: 7


## Step 4: Scaling and transforming numerical variables

A power transformation (Yeo-Johnson) is performed on the Amount variable to overcome its high right-skewness, and Robust Scaler on the Time variable, which uses a statistic that is robust to outliers, i.e., median and interquartile range. Both are fitted on only training data and the transformed test data using fitted values of training data (so that test data's Amount and Time values don't influence the transformation). There are no categorical variables in this dataset to be encoded.

In [6]:
X_train_raw = X_train_raw.copy()
X_test_raw = X_test_raw.copy()

pt = PowerTransformer(method='yeo-johnson')
X_train_raw['Amount_transformed'] = pt.fit_transform(X_train_raw[['Amount']])
X_test_raw['Amount_transformed'] = pt.transform(X_test_raw[['Amount']])  # transform only, not fit

rs = RobustScaler()
X_train_raw['Time_scaled'] = rs.fit_transform(X_train_raw[['Time']])
X_test_raw['Time_scaled'] = rs.transform(X_test_raw[['Time']])  # transform only, not fit

print(f"Amount skew before: {X_train_raw['Amount'].skew():.3f}, after (train): {X_train_raw['Amount_transformed'].skew():.3f}")

Amount skew before: 10.572, after (train): 0.019


## Step 5: Feature selection

The correlation analysis from Task 4, Phase 1, indicates that linear signal relating to fraud is sequestered within a few PCA components and that most of the other 20+ features contribute very little. This finding is even more salient to the LCS than it is for most other models; the eLCS rule population is tasked with matching an attribute set across all of included features, thus superfluous features are likely increasing the size of the search space, slowing convergence and diluting the specificity of the rules without adding any predictive benefit. Again, correlation ranking is calculated based only on the training set (y_train), which prevents data leakage into the feature selection. The top 10 features with the highest absolute correlation to the Class from the training data are chosen, reducing the attributes from 30 down to 10 before rule evolution begins.

In [7]:
all_feature_cols = ['Time_scaled'] + [f'V{i}' for i in range(1, 29)] + ['Amount_transformed']

train_with_target = X_train_raw[all_feature_cols].copy()
train_with_target['Class'] = y_train.values

corr = train_with_target.corr()['Class'].drop('Class').abs().sort_values(ascending=False)
top_features = corr.head(10).index.tolist()

print("Selected features (top 10 by |correlation| with Class, computed on the training set only):")
print(corr.head(10))

Selected features (top 10 by |correlation| with Class, computed on the training set only):
V17    0.376918
V12    0.246406
V14    0.243894
V7     0.227940
V10    0.220210
V16    0.211398
V3     0.209367
V5     0.148035
V18    0.141935
V1     0.140415
Name: Class, dtype: float64


## Step 6: Discretisation

The eLCS can handle continuous attributes natively, but LCS rules are naturally better suited to discrete categorical-type conditions as saying "feature is bin 3" is more intuitive and arguably more conducive for the genetic algorithm's discovery of specialised rules than aiming for an arbitrary continuous range. The 10 features identified are each discretised into 5 quantile-based bins using KBins Discretizer fitted only on the training set. Applying a fitted KBins Discretizer to the test set ensures the bin boundaries are defined by training-set quantiles, not test-set. Quantile-based binning is chosen over equal-width binning for this task because many features are not uniformly distributed-for instance, V17 and V14 would likely result in sparsely populated bins under equal-width binning.

In [8]:
kbd = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')

X_train_disc = kbd.fit_transform(X_train_raw[top_features])
X_test_disc = kbd.transform(X_test_raw[top_features])  # transform only, not fit

X_train_final = pd.DataFrame(X_train_disc, columns=top_features)
X_test_final = pd.DataFrame(X_test_disc, columns=top_features)

print(X_train_final.head())
print(f"\nUnique bins per feature (train): \n{X_train_final.nunique()}")

   V17  V12  V14   V7  V10  V16   V3   V5  V18   V1
0  0.0  2.0  2.0  1.0  2.0  3.0  3.0  4.0  1.0  0.0
1  1.0  4.0  3.0  3.0  0.0  2.0  4.0  3.0  2.0  0.0
2  0.0  4.0  3.0  3.0  1.0  3.0  2.0  3.0  1.0  2.0
3  2.0  0.0  4.0  4.0  4.0  0.0  0.0  4.0  2.0  1.0
4  4.0  3.0  4.0  3.0  3.0  0.0  3.0  2.0  2.0  2.0

Unique bins per feature (train): 
V17    5
V12    5
V14    5
V7     5
V10    5
V16    5
V3     5
V5     5
V18    5
V1     5
dtype: int64


## Step 7: Class imbalance
Baseline results for Phase 2's Task 2 highlight the importance of using such an approach with an LCS: the scarcity of fraud examples in the training set meant that the eLCS's population of rules rarely experienced minority class instances to allow for rules that generalize better than specific examples of fraud. SMOTE alleviates this problem by synthetically constructing examples of fraud within the feature space rather than replicating existing cases. Thus the rule discovery mechanism has access to significantly more and varied fraud examples upon which it can operate. SMOTE is only applied to the training set as a last pre-processing step. Therefore, the test set retains the original class proportion.

In [9]:
print(f"Before SMOTE - Training set: {X_train_final.shape}, Fraud cases: {y_train.sum()}")

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_final, y_train)

print(f"After SMOTE  - Training set: {X_train_bal.shape}, Fraud cases: {y_train_bal.sum()}")
print(f"Test set (untouched): {X_test_final.shape}, Fraud cases: {y_test.sum()}")

Before SMOTE - Training set: (16000, 10), Fraud cases: 26
After SMOTE  - Training set: (31948, 10), Fraud cases: 15974
Test set (untouched): (4000, 10), Fraud cases: 7


## Step 8: Sanity check — running eLCS on the preprocessed data

This is not yet the improved system from Task 4 (no eLCS hyperparameters have been changed here), but running the unmodified model on this preprocessed, feature-selected, discretised, and class-balanced data gives an early read on whether preprocessing alone moves the needle before any algorithmic improvements are made in Task 4.

In [10]:
model = eLCS(learning_iterations=10000, random_state=42)

start = time.time()
model.fit(X_train_bal.values, y_train_bal.values)
elapsed = time.time() - start

preds = model.predict(X_test_final.values)

print(f"Training time: {elapsed:.2f} seconds")
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, preds):.4f}")
print(f"Precision: {precision_score(y_test, preds, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, preds, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, preds, zero_division=0):.4f}")

Training time: 16.68 seconds
Accuracy: 0.9988
Balanced Accuracy: 0.9994
Precision: 0.5833
Recall: 1.0000
F1 Score: 0.7368


### Task 3 Results: Data Preprocessing and Feature Engineering
The preprocessing began with the raw data, and 1,081 duplicate rows that were previously found in Task 2 were removed, bringing the row count from 284,807 down to 283,726. The data contained no missing, invalid, or inconsistent values, therefore, no fixes were needed. The outliers in 'Amount' were kept, once again keeping consistency with the previous task, because removing the outliers potentially would remove actual instances of fraud. The outliers were essentially representing varying amounts of consumer spending and not incorrect or poor data.

As with Task 2, a subsample of 20,000 rows was taken to make the training computationally tractable. This subset contained 33 cases of fraud and 19,967 cases that were legitimate, which kept the fraud rate the same as the original dataset. Unlike Task 2, the data was first partitioned into training (16,000 rows and 26 cases of fraud) and testing (4,000 rows and 7 cases of fraud) subsets before any of the data parameters were fitted to the data. All the steps taken during preprocessing were fitted only on the training set and the learned parameters were then applied to the test set to avoid any data leakage.

A Yeo-Johnson power transform was performed on the 'Amount' variable which reduced the skew on the training set from 10.572 down to 0.019. The 'Time' variable was then scaled using a Robust Scaler, both of these fitted on the training data. The feature space was reduced from 30 down to the 10 most correlated features with the 'Class' variable (V17, V12, V14, V7, V10, V16, V3, V5, V18, V1) on the training set only, this helped to minimize the search space of the LCS's rule population. These 10 features were each split into 5 bins of equal quantity using the KBins Discretizer, which was also fitted on the training set, as discrete bins of features work better with the rule conditions of LCSs than continuous ranges.

Lastly, the SMOTE oversampling technique was used on the training set, after the other pre-processing steps, and this increased the number of cases of fraud from 26 up to 15,974 bringing the training set size to 31,948 total rows. The 4,000 row test set remained untouched and held the original proportion of fraudulent instances to use for final validation.

When the same unmodified eLCS config from Task 2 (10,000 learning_iterations, all default parameters, and no code changes) was used on this preprocessed data, these are the results produced:

**Preprocessed results**:
|  Metric  |  Value  |  
|----------|---------|  
| Accuracy | 0.9988  |  
| Balanced Accuracy | 0.9994 |  
| Precision | 0.5833  |  
| Recall   | 1.0000  |  
| F1-score | 0.7368  |  

Compared to the baseline of Task 2, performance significantly improves where it really counts. Balanced accuracy jumped from just better than random (0.5714) to near perfect (0.9994) and recall improved from a miserable 1 out of 7 fraud cases to 7 out of 7. The trade off for this improvement in recall is a drop in precision from 1.0000 to 0.5833, meaning that the model now identifies some legitimate transactions as fraud. 

This is the expected trade-off that occurs when tackling severe class imbalance: by exposing the rule discovery process to more of the minority class, it is allowed it to form more comprehensive fraud rules, which occasionally also flag innocent transactions. 

Assuming the cost of missing a fraud event is higher than the cost of a false alarm, this is almost certainly a change in the right direction. Task 4's improvements should focus on fine tuning this precision/recall trade-off.